# NetMedGPT vs TxGNN by size of the disease neighbourhood (rare disease split)

A message-passing GNN builds a node's representation from its neighbourhood; when a disease has few neighbours, little
information is available to aggregate. NetMedGPT learns relational patterns from pseudo-sentences across the whole graph.
If this matters, NetMedGPT's advantage over TxGNN should be largest for diseases with small neighbourhoods.

- Pairs: the test pairs of the rare disease split (`netmedgpt_eval_pairs.csv`), scored by both models
  (TxGNN: `pair_scores.csv` of eval_txgnn_netmedgpt_metrics.py; NetMedGPT: saved checkpoints, as in train.py).
- Each test disease gets the number of nodes within 2 hops in the training graph of the split; the diseases are split
  into three equal groups (small / medium / large).
- Per group: AUROC / AUPRC of all its pairs and the per-disease AUROC (positives vs negatives of the same disease,
  averaged over diseases); paired t-test over 5 seeds.

In [ ]:
import os
import sys
if os.path.basename(os.getcwd()) == "revision_scripts":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
import json
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy import stats
from sklearn.metrics import roc_auc_score, average_precision_score
import torch

from netmedgpt.model import TransformerModel
from netmedgpt.prepare_txgnn_splits import prepare_txgnn_data
from netmedgpt.utilities import sim

SEEDS = [1, 2, 3, 4, 5]
RELATIONS = ["indication", "contraindication", "off-label use"]
TXGNN_SCORES = "/home/bbc8731/All_NetMedGPT/txgnn_docker/TxGNN_github/reproduce/txgnn_rare_disease_seed{seed}/pair_scores.csv"
TXGNN_FALLBACK = "data/result/txgnn_rare_disease_seed{seed}/pair_scores.csv"
CHECKPOINT = "data/model_checkpoints/saved_models_evaluation/netmedgpt_rare_disease_seed{seed}.pt"
GPU = 0
OUT_DIR = "data/new_analysis/neighbourhood_rare"
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# ── KG and model setup (as other/evaluate_checkpoints.py) ─────────────────────
all_param = json.load(open("data/parameters.json"))
nodes = pd.read_csv("data/KG/nodes.csv")
edge = pd.read_csv("data/KG/edges.csv")
feat = torch.load("data/KG/embeddings.pt")
n_nodes = len(nodes)

mask_token = int(edge["z_index"].max()) + 1
entity = list(nodes["node_type"].unique()) + list(edge["relation"].unique()) + ["mask"]
relation_mask_index = pd.concat([edge[["relation", "z_index"]].drop_duplicates(),
                                 pd.DataFrame([["mask", mask_token]], columns=["relation", "z_index"])], ignore_index=True)
rel_token = {r: int(edge.loc[edge["relation"] == r, "z_index"].iloc[0]) for r in RELATIONS}
hp = all_param["best_hyperparam"]
seq_len = all_param["node2vec"]["walk_length"] * 2 - 1
device = torch.device(f"cuda:{GPU}" if torch.cuda.is_available() else "cpu")


def load_model(seed):
    ckpt = torch.load(CHECKPOINT.format(seed=seed), map_location=device)
    state_dict = ckpt["model_state"] if "model_state" in ckpt else ckpt     # two saved formats
    model = TransformerModel(mask_token + 1, hp["hidden_channels"], hp["nhead"], hp["N_encoder_layers"], seq_len,
                             device=device, feat=feat, nodes=nodes, entity=entity,
                             relation_mask_index=relation_mask_index, pos_emb="fixed").to(device)
    model.load_state_dict(state_dict)
    return model.eval()


def training_graph(seed):
    """undirected adjacency of the training edges of the rare disease split"""
    split_dir = f"data/splits/rare_disease_{seed}"
    splits = [pd.read_csv(os.path.join(split_dir, f), low_memory=False) for f in ["train.csv", "valid.csv", "test.csv"]]
    train, _, _ = prepare_txgnn_data(nodes, *splits, {r: "disease|drug" for r in RELATIONS})
    x, y = train["x_index"].to_numpy(), train["y_index"].to_numpy()
    A = sp.coo_matrix((np.ones(2 * len(x), dtype=np.int8), (np.r_[x, y], np.r_[y, x])), shape=(n_nodes, n_nodes))
    return A.tocsr()



In [ ]:
# ── Scores of the test pairs, per seed ───────────── ─────────────────────────
all_pairs = []
for seed in SEEDS:
    path = TXGNN_SCORES.format(seed=seed)
    tx = pd.read_csv(path if os.path.exists(path) else TXGNN_FALLBACK.format(seed=seed))
    tx = tx[tx["split"] == "test"].reset_index(drop=True)

    model = load_model(seed)
    idx = torch.tensor(np.vstack([tx["disease_index"], tx["relation"].map(rel_token), tx["drug_index"]]), dtype=torch.long)
    tx["netmedgpt"] = sim(idx, model, device=device, mask_token=mask_token, seq_len=seq_len,
                          batch_size=hp["batch_size"], method="prob")
    tx["txgnn"] = tx["score"]
    del model
    torch.cuda.empty_cache()

    tx["seed"] = seed
    all_pairs.append(tx[["seed", "relation", "disease_index", "drug_index", "label", "netmedgpt", "txgnn"]])

    # check: overall AUROC must match the saved results (same pairs, same scores)
    for rel, g in tx.groupby("relation"):
        print(f"seed {seed} {rel:17s} AUROC NetMedGPT {roc_auc_score(g.label, g.netmedgpt):.4f} | TxGNN {roc_auc_score(g.label, g.txgnn):.4f}")

pairs = pd.concat(all_pairs, ignore_index=True)
pairs.to_csv(os.path.join(OUT_DIR, "pairs.csv"), index=False)


In [ ]:
pairs = pd.read_csv(os.path.join(OUT_DIR, "pairs.csv"))
METHODS = ["netmedgpt", "txgnn"]
GROUPS = ["small", "medium", "large"]


def per_disease_auroc(g, m):
    """mean over diseases of the AUROC of their positives vs their negatives"""
    v = [roc_auc_score(d.label, d[m]) for _, d in g.groupby("disease_index") if d.label.nunique() == 2]
    return np.mean(v) if v else np.nan


# 2-hop neighbourhood size of every test disease, per seed (training graph of that seed)
hood = []
for seed in SEEDS:
    A = training_graph(seed)
    A.data[:] = 1
    diseases = pairs.loc[pairs.seed == seed, "disease_index"].unique()
    R = A[diseases]
    two_hop = ((R + R @ A) > 0).sum(axis=1).A1
    hood.append(pd.DataFrame({"seed": seed, "disease_index": diseases, "degree": R.getnnz(axis=1), "hood_2hop": two_hop}))
hood = pd.concat(hood, ignore_index=True)
pairs_h = pairs.merge(hood, on=["seed", "disease_index"])
pairs_h["hood_group"] = pairs_h.groupby("seed")["hood_2hop"].transform(lambda x: pd.qcut(x.rank(method="first"), 3, labels=GROUPS))

rows = []
for (seed, rel, grp), g in pairs_h.groupby(["seed", "relation", "hood_group"], observed=True):
    for m in METHODS:
        rows.append({"seed": seed, "relation": rel, "neighbourhood": grp, "method": m,
                     "median 2-hop nodes": g.drop_duplicates("disease_index")["hood_2hop"].median(),
                     "AUROC": roc_auc_score(g.label, g[m]), "AUPRC": average_precision_score(g.label, g[m]),
                     "per-disease AUROC": per_disease_auroc(g, m)})
t1 = pd.DataFrame(rows)
t1.to_csv(os.path.join(OUT_DIR, "test1_neighbourhood_per_seed.csv"), index=False)

summary1 = []
for (rel, grp), g in t1.groupby(["relation", "neighbourhood"], sort=False):
    r = {"relation": rel, "neighbourhood": grp, "median 2-hop nodes": g["median 2-hop nodes"].median()}
    for metric in ["AUROC", "AUPRC", "per-disease AUROC"]:
        w = g.pivot_table(index="seed", columns="method", values=metric)
        r[f"{metric} NetMedGPT"], r[f"{metric} TxGNN"] = w["netmedgpt"].mean(), w["txgnn"].mean()
        r[f"{metric} p"] = stats.ttest_rel(w["netmedgpt"], w["txgnn"]).pvalue
    summary1.append(r)
summary1 = pd.DataFrame(summary1)
summary1["neighbourhood"] = pd.Categorical(summary1["neighbourhood"], GROUPS, ordered=True)
summary1 = summary1.sort_values(["relation", "neighbourhood"])
summary1.to_csv(os.path.join(OUT_DIR, "test1_neighbourhood_summary.csv"), index=False)

# change from small to large neighbourhoods per method (negative = performance drops in crowded neighbourhoods)
drop = (t1.pivot_table(index=["seed", "relation", "method"], columns="neighbourhood", values="per-disease AUROC", observed=True)
          .eval("large - small").rename("large - small").reset_index()
          .pivot_table(index="relation", columns="method", values="large - small"))
print("Per-disease AUROC, large minus small neighbourhoods (mean over seeds):")
print(drop.round(3))
summary1.round(3)